In [1]:
import numpy as np
import mvt_estimation
import pandas as pd

In [2]:
data = pd.read_excel("data/techstocks.xlsx")

In [9]:
df = data[['AAPL', 'MSFT', 'AMZN', 'GOOG', 'TSLA']].to_numpy()
theta, VCV_sam, VCV_asy, _ = mvt_estimation.MVT_MLE_approach2(df)

In [13]:
def defining_theta(theta, p):
    theta = np.asarray(theta).ravel()
    mu = theta[:p]
    nu = theta[-1]
    a = theta[p:-1]

    Sigma = np.zeros((p, p))
    idx = np.tril_indices(p)
    Sigma[idx] = a
    Sigma = Sigma + Sigma.T - np.diag(np.diag(Sigma))
    return mu, Sigma, nu


def report_mvt_results(theta_hat, VCV_asy, X, names=None):
    n, p = X.shape
    if names is None:
        names = [f"X{j+1}" for j in range(p)]

    se = np.sqrt(np.diag(VCV_asy))

    mu_hat, Sigma_hat, nu_hat = defining_theta(theta_hat, p)

    se_mu = se[:p]
    se_nu = se[-1]

    cov_implied = (nu_hat / (nu_hat - 2.0)) * Sigma_hat
    vol = np.sqrt(np.diag(cov_implied))

    corr_implied = cov_implied / np.outer(vol, vol)

    C = corr_implied.copy()
    np.fill_diagonal(C, -np.inf)
    i, j = np.unravel_index(np.argmax(C), C.shape)

    print("\nMVT MLE estimates (with asymptotic SE)")
    for nm, m, s in zip(names, mu_hat, se_mu):
        print(f"{nm:5s}: mu_hat = {m: .6f}   SE = {s: .6f}   t = {m/s: .3f}")

    print(f"\nnu_hat = {nu_hat:.6f}   SE = {se_nu:.6f}   t = {nu_hat/se_nu: .3f}")

    print("\nImplied daily volatility (from Cov = nu/(nu-2)*Sigma)")
    for nm, v in zip(names, vol):
        print(f"{nm:5s}: {v:.4f}%")

    print("\nImplied correlation matrix (rounded)")
    print(np.round(corr_implied, 3))

    print(f"\nHighest correlation pair: {names[i]} - {names[j]} = {corr_implied[i, j]:.3f}")

    se_sig_vech = se[p:-1]
    print("\nSigma_hat (scale matrix in MVT)")
    print(Sigma_hat)

    print("\nSEs for vech(Sigma) (lower triangle incl diag)")
    k = 0
    for r in range(p):
        for c in range(r + 1):
            print(f"Sigma[{r+1},{c+1}] = {Sigma_hat[r,c]: .6f}   SE = {se_sig_vech[k]: .6f}")
            k += 1

    return {
        "mu_hat": mu_hat,
        "Sigma_hat": Sigma_hat,
        "nu_hat": nu_hat,
        "VCV_asy": VCV_asy,
        "se": se,
        "cov_implied": cov_implied,
        "corr_implied": corr_implied,
        "vol": vol,
    }

In [15]:
names = ["AAPL", "MSFT", "AMZN", "GOOG", "TSLA"]
out = report_mvt_results(theta, VCV_sam, df, names=names)


MVT MLE estimates (with asymptotic SE)
AAPL : mu_hat =  0.105696   SE =  0.051961   t =  2.034
MSFT : mu_hat =  0.115778   SE =  0.045990   t =  2.517
AMZN : mu_hat =  0.096225   SE =  0.064304   t =  1.496
GOOG : mu_hat =  0.196602   SE =  0.060185   t =  3.267
TSLA : mu_hat =  0.071318   SE =  0.130867   t =  0.545

nu_hat = 4.352906   SE = 0.372345   t =  11.691

Implied daily volatility (from Cov = nu/(nu-2)*Sigma)
AAPL : 1.6079%
MSFT : 1.4153%
AMZN : 1.9778%
GOOG : 1.8540%
TSLA : 4.0263%

Implied correlation matrix (rounded)
[[1.    0.462 0.43  0.429 0.388]
 [0.462 1.    0.642 0.531 0.395]
 [0.43  0.642 1.    0.585 0.411]
 [0.429 0.531 0.585 1.    0.41 ]
 [0.388 0.395 0.411 0.41  1.   ]]

Highest correlation pair: MSFT - AMZN = 0.642

Sigma_hat (scale matrix in MVT)
[[1.39747311 0.56875709 0.738782   0.69086208 1.35675966]
 [0.56875709 1.08275928 0.97148747 0.75287856 1.2174764 ]
 [0.738782   0.97148747 2.11441929 1.15925619 1.76860705]
 [0.69086208 0.75287856 1.15925619 1.858075